In [ ]:
import torch
import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", 
                                            device_map="cpu", 
                                            dtype=torch.bfloat16,)

# Example: Create sample data
# time_steps = 300  # 5 minutes of data
# hr_data = np.random.randint(60, 100, size=time_steps).astype(np.float32)
# spo2_data = np.random.randint(90, 100, size=time_steps).astype(np.float32)

# Convert to tensor - each series as a batch item
hr_tensor = torch.tensor(hr_data).unsqueeze(0)  # (1, 300)
spo2_tensor = torch.tensor(spo2_data).unsqueeze(0)  # (1, 300)

# Combine into batch: (2, 300)
context = torch.stack([hr_tensor, spo2_tensor]).squeeze(1)  # (2, 300)

# Get embeddings using the pipeline's internal method
with torch.no_grad():
    # Method 1: Try embed method if available
    if hasattr(pipeline, 'embed'):
        embeddings = pipeline.embed(context)

print(f"Input shape: {context.shape}")
print(f"Embeddings shape: {embeddings}")
print(f"First few embeddings:\n{embeddings[0, :3, :5]}")  # Show sample

/Users/adityagoyal/miniconda3/envs/baseline_env/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/Users/adityagoyal/miniconda3/envs/baseline_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import scipy.io
import torch
import numpy as np

def get_embeddings_from_mat(mat_file_path, pipeline, window_size=300):
    """Extract embeddings from MAT file - simplified"""
    # Load data
    mat_data = scipy.io.loadmat(mat_file_path)
    hr_cols = mat_data['vdata'][:,0]
    spo2_cols = mat_data['vdata'][:,2]
    
    # Filter valid data
    valid_mask = ~np.isnan(hr_cols) & ~np.isnan(spo2_cols)
    hr_valid = hr_cols[valid_mask].astype(np.float32)[:window_size]
    spo2_valid = spo2_cols[valid_mask].astype(np.float32)[:window_size]
    
    # Convert to tensors and stack
    hr_tensor = torch.tensor(hr_valid, dtype=torch.float32)
    spo2_tensor = torch.tensor(spo2_valid, dtype=torch.float32)
    context = torch.stack([hr_tensor, spo2_tensor])  # (2, window_size)
    
    # Get embeddings - that's it!
    with torch.no_grad():
        embeddings = pipeline.embed(context)
    
    return embeddings

# Test it
mat_file = '/Users/adityagoyal/Desktop/Research - yin li/EDA/data/download/Vitals/NICU_1005_vitals.mat'
embeddings = get_embeddings_from_mat(mat_file, pipeline)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[-1]}")